# 08 - Evaluation Framework

Thesis-grade evaluation of the **hybrid sparse-dense-graph + GraphRAG** retrieval
stack.  Two strictly separated layers (per the "Separate Retrieval from Generation" requirement):

* **Layer A - retrieval quality** - did we retrieve the correct evidence?
* **Layer B - answer quality** - did the LLM use that evidence to produce a
  correct, grounded answer?

plus overlap / context / triangulation, statistics, the four thesis tables,
figures, a reproducibility report, and **constructed compliance scenarios**
(dual-drafter -> blind adjudicator -> dispute-escalation protocol) with the
four RAG baselines.

> Every tunable lives in `src/evaluation/config.py` (`EvalConfig`, `EXPERIMENTS`).
> All outputs land under `notebooks/data/evaluation/` and
> `notebooks/data/figures/`.  All LLM calls are deterministic + cached.


In [1]:
import sys, os, json, warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")   # urllib3 / requests version noise
ROOT = Path.cwd()                   # execution from repo root (see launch)
SRC  = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from evaluation import config as E, benchmark as BM
import evaluation.retrieval_run   as RR
import evaluation.overlap         as OV
import evaluation.context         as CTX
import evaluation.generation      as GEN
import evaluation.answer_metrics  as AM
import evaluation.error_analysis  as EA
import evaluation.stats           as ST
import evaluation.tables          as TB
import evaluation.figures         as FIG
import evaluation.report          as RP
import evaluation.scenarios       as SC
import evaluation.baselines       as BL
import evaluation.semantic        as SEM
import common
import csv
import numpy as np
from retrieval._corpus import load_corpus

cfg = E.EvalConfig()
print("[setup] repo root :", ROOT)
print("[setup] llm/judge :", cfg.llm_model, "/", cfg.judge_model)
print("[setup] scenario  :",
      cfg.scenario_drafter_a, "+", cfg.scenario_drafter_b,
      "adjudicator=", cfg.scenario_adjudicator,
      "escalator=", cfg.scenario_escalator)
print("[setup] k-values  :", list(cfg.k_values), " retrieve_k=", cfg.retrieve_k)


[setup] repo root : <repo>
[setup] llm/judge : qwen3.8:27b / qwen3.8:27b
[setup] scenario  : gemma3:27b + mistral-small3.1:24b adjudicator= qwen3.8:27b escalator= nemotron-3-nano:30b
[setup] k-values  : [1, 3, 5, 10, 20]  retrieve_k= 50


## 1. Benchmark (per "Benchmark Interface") - 60-query gold set

In [2]:
items = BM.build_benchmark()          # deterministic, Random(7), identical to notebook 05
BM.save(items, E.OUT_PER_QUERY / "benchmark.jsonl")
s = BM.summary(items)
print("n_queries :", s["n"])
print("categories:", s["category_counts"])
print("n_docs    :", s["n_docs"])


n_queries : 60
categories: {'article': 58, 'term': 2}
n_docs    : 16
leakage   : {'targets_that_are_lora_training_positives': 49, 'questions_that_match_training_queries': 2, 'note': "data-leakage: the LoRA encoder was trained on positive pairs whose target is this gold item's target. Any dense-finetune row benefiting from this is a known inflation; the base-encoder rows are the clean baseline."}


## 2. Layer A - retrieval quality (per "Retrieval Metrics" through "Graph + Hybrid Evaluation", and "Evaluation Configuration Matrix")

All `EXPERIMENTS` system over the shared benchmark, same k-set, same relevance
judgments (the "Methodological Neutrality" requirement).  `neo4j` / `hybrid_graph` need the live Neo4j; they are skipped gracefully if down.


In [3]:
eng = RR.RetrievalEngine()
retrieval = RR.run_retrieval(items)        # all 8 systems
print("per-system row counts:")
for sname, rows in retrieval.items():
    print(f"  {sname:<14} {len(rows)}")
skip = E.OUT_PER_QUERY / "skipped_systems.json"
if skip.exists():
    print("SKIPPED:", json.loads(skip.read_text()))
agg = TB.load_aggregate()
print("aggregate rows:", len(agg))


per-system row counts:
  dense          60
  sparse         60
  hybrid         60
  graph          60
  neo4j          60
  hybrid_graph   60
  dense_ft_s1    60
  dense_ft_s2    60
aggregate rows: 200


## 3. Retrieval overlap + complementarity (per "Retrieval Overlap Analysis")

Jaccard between methods' top-k and *unique* relevant evidence each method
surfaces (the thesis's evidence for retrieval **complementarity**), incl.
targets found only after hybridization.

In [4]:
per_query_by_system = {
    s: {r["query_id"]: [{"lineage_id": lid} for lid in r["retrieved_top20"]]
        for r in rows}
    for s, rows in retrieval.items()
}
OV_SYSTEMS = ["dense", "sparse", "graph", "hybrid"]
K_OV = 5
ov   = OV.overlap_matrix(per_query_by_system, OV_SYSTEMS, K_OV)
comp = OV.complementarity(per_query_by_system, items, OV_SYSTEMS + ["hybrid"], K_OV)
OV.save(ov, comp)
print("Jaccard@%d:\n" % K_OV, json.dumps(ov, indent=2))
print("complementarity:", json.dumps(comp, indent=2))


Jaccard@5:
 {
  "dense~sparse": 0.2519,
  "dense~graph": 0.2423,
  "dense~hybrid": 0.4787,
  "sparse~graph": 0.8023,
  "sparse~hybrid": 0.5061,
  "graph~hybrid": 0.4662
}
complementarity: {
  "k": 5,
  "unique_relevant_only_by": {
    "dense": 3,
    "sparse": 0,
    "graph": 0
  },
  "queries_with_no_method_hitting_target": 2,
  "targets_found_only_after_fusion": 0,
  "core_systems": [
    "dense",
    "sparse",
    "graph"
  ]
}


## 4. Context coverage + triage (per "RAG Triangulation" - context level)

Does the retrieved target actually make it into the top-5 context window the
LLM is conditioned on, per system?

In [5]:
ctx_rows = []
for item in items:
    for sname, rows in retrieval.items():
        pqr = next((r for r in rows if r["query_id"] == item.query_id), None)
        if not pqr:
            continue
        ctx_rows.append({"system": sname, "query_id": item.query_id,
                         **CTX.context_coverage(pqr, item)})
CTXT = CTX.save(ctx_rows)
in_win = [r for r in ctx_rows if r["in_context_window"]]
print("rows:", len(ctx_rows), "| target-in-window:", len(in_win),
      "(%.1f%%)" % (100*len(in_win)/max(len(ctx_rows),1)))
agg_by = Counter()
den = Counter()
for r in ctx_rows:
    den[r["system"]] += 1
    agg_by[r["system"]] += int(r["in_context_window"])
for sname, d in den.items():
    print(f"  {sname:<14} {agg_by[sname]/d:.2f}")


rows: 480 | target-in-window: 370 (77.1%)
  dense          0.80
  sparse         0.92
  hybrid         0.92
  graph          0.92
  neo4j          0.22
  hybrid_graph   0.92
  dense_ft_s1    0.77
  dense_ft_s2    0.72


## 5. Layer B - answer generation (per "End-to-End RAG Evaluation" and "Methodological Neutrality")

The **same** LLM (`qwen3.8:27b`, temp 0.0, max_tokens 2048, context window 5)
answers every query for every system - only the retrieved evidence varies, so
comparisons are on an identical generation footing.  Neo4j-backed system is
run last and isolated so a DB outage does not lose the first three.

In [6]:
GEN_SYSTEMS = ["sparse", "dense", "hybrid", "neo4j"]
gen = {}
for sname in GEN_SYSTEMS:
    try:
        out = GEN.run_generation(items, retrieval, systems=[sname])
        gen.update(out)
        print(f"[gen] {sname:<10} OK  rows={len(out.get(sname, []))}")
    except Exception as e:
        print(f"[gen] {sname:<10} FAILED {type(e).__name__}: {e}")
print("systems with generations:", list(gen))


[gen] sparse     OK  rows=60


[gen] dense      OK  rows=60


[gen] hybrid     OK  rows=60


[gen] neo4j      OK  rows=60
systems with generations: ['sparse', 'dense', 'hybrid', 'neo4j']


## 6. Answer metrics (per "Answer-Level Metrics")

EM / token-F1 / semantic similarity (bge-m3) / target-citation / 4-axis
LLM-as-judge rubric (1-5).  Same-model-as-generator caveat is reported.

In [7]:
scored = AM.run(items, gen)
ans_agg = AM.aggregate(scored)
E.OUT_AGGREGATE.mkdir(parents=True, exist_ok=True)
with open(E.OUT_AGGREGATE / "answer_aggregate.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["system", "metric", "value", "n"])
    w.writeheader(); w.writerows(ans_agg)
for r in ans_agg:
    print(f"{r['system']:<10} {r['metric']:<22} {r['value']:.4f} (n={r['n']})")


sparse     cites_target           1.0000 (n=60)
sparse     exact_match            0.0000 (n=60)
sparse     judge__completeness    4.5333 (n=60)
sparse     judge__faithfulness    4.8500 (n=60)
sparse     judge__groundedness    4.8833 (n=60)
sparse     judge__relevance       4.8833 (n=60)
sparse     semantic_similarity    0.8638 (n=60)
sparse     token_f1               0.5997 (n=60)
dense      cites_target           1.0000 (n=60)
dense      exact_match            0.0000 (n=60)
dense      judge__completeness    4.6000 (n=60)
dense      judge__faithfulness    4.9000 (n=60)
dense      judge__groundedness    4.8167 (n=60)
dense      judge__relevance       4.7333 (n=60)
dense      semantic_similarity    0.8423 (n=60)
dense      token_f1               0.5899 (n=60)
hybrid     cites_target           1.0000 (n=60)
hybrid     exact_match            0.0000 (n=60)
hybrid     judge__completeness    4.6000 (n=60)
hybrid     judge__faithfulness    4.8000 (n=60)
hybrid     judge__groundedness    4.8333

## 7. Statistics (per "Bootstrap Confidence Intervals") - bootstrap CIs + paired tests

Pairwise `hybrid` vs each single method at top-10 on per-query recall, with a
2000-resample paired bootstrap CI, Wilcoxon signed-rank, and Cohen's d.

In [8]:
def _perq(system, metric, k):
    byq = {}
    for r in retrieval.get(system, []):
        v = (r["metrics"].get(str(k), {}) or {}).get(metric)
        if v is not None:
            byq[r["query_id"]] = v
    return byq

def _pair(sys_a, sys_b, metric, k):
    a, b = _perq(sys_a, metric, k), _perq(sys_b, metric, k)
    common = [q for q in a if q in b]
    va = [a[q] for q in common]; vb = [b[q] for q in common]
    return ST.summarize_comparison(va, vb, sys_a, sys_b)

COMPARISONS = [("hybrid", m, "recall", 10) for m in ("dense", "sparse", "graph")]
stat_rows = []
for (a, b, m, k) in COMPARISONS:
    try:
        c = _pair(a, b, m, k); stat_rows.append(c); print(c)
    except Exception as e:
        print(f"{a} vs {b}: {type(e).__name__}: {e}")
json.dump(stat_rows, open(E.OUT_AGGREGATE / "stats_comparison.json", "w"), indent=2)


{'a': 'hybrid', 'b': 'dense', 'mean_a': 0.9333333333333333, 'mean_b': 0.8333333333333334, 'diff_a_minus_b': 0.1, 'ci95': [0.03333333333333333, 0.18333333333333332], 'wilcoxon_p': 0.014305878435429648, 'cohens_d': 0.3305438840143004, 'n': 60}
{'a': 'hybrid', 'b': 'sparse', 'mean_a': 0.9333333333333333, 'mean_b': 0.9333333333333333, 'diff_a_minus_b': 0.0, 'ci95': [-0.06666666666666667, 0.06666666666666667], 'wilcoxon_p': 1.0, 'cohens_d': 0.0, 'n': 60}
{'a': 'hybrid', 'b': 'graph', 'mean_a': 0.9333333333333333, 'mean_b': 0.9333333333333333, 'diff_a_minus_b': 0.0, 'ci95': [-0.06666666666666667, 0.06666666666666667], 'wilcoxon_p': 1.0, 'cohens_d': 0.0, 'n': 60}


## 8. Error analysis

Failure taxonomy per (system, query). Labels are routing aids, not verified verdicts.


In [9]:
classification = {item.query_id: EA.classify(item, retrieval, gen, k=5)
                  for item in items}
EA.save(classification)
labels = Counter(v for d in classification.values() for v in d.values())
print("label distribution:", dict(labels))


label distribution: {'GENERATION_FAILURE': 171, 'CORRECT': 199, 'RETRIEVAL_FAILURE': 60, 'PARTIAL_RETRIEVAL': 50}
leakage counts: {'question_verbatim_in_training': 2, 'targets_that_are_training_positives': 49, 'reference_text_in_training_positives': 49, 'n_queries': 60}
risk note: LoRA dense encoders were trained ON the gold-set targets (49/60). Finetuned-dense rows are therefore inflated vs a clean baseline; the base-encoder rows are the unbiased estimate. Report both and flag the leakage rather than presenting the finetuned number as a clean win.


## 9. Thesis tables (per "Hybrid Retrieval Contribution Analysis" and "GraphRAG Evaluation")

A - retrieval per system; B - answer quality; plus the hybrid-vs-baseline comparison.


In [10]:
tA = TB.table_a(agg)
tB = TB.table_b(ans_agg)
tComp = TB.comparison(agg)
TB.write({"A": tA, "B": tB, "comparison": tComp})
for name, t in [("A", tA), ("B", tB), ("comparison", tComp)]:
    print(f"\n### {name}")
    for r in t:
        print("   " + json.dumps(r))



### Table A
   {"system": "dense", "k": 1, "recall": "0.400", "precision": "0.400", "hit": "0.400", "mrr": "0.400", "ndcg": "0.400"}
   {"system": "dense", "k": 3, "recall": "0.700", "precision": "0.233", "hit": "0.700", "mrr": "0.533", "ndcg": "0.576"}
   {"system": "dense", "k": 5, "recall": "0.800", "precision": "0.160", "hit": "0.800", "mrr": "0.556", "ndcg": "0.617"}
   {"system": "dense", "k": 10, "recall": "0.833", "precision": "0.083", "hit": "0.833", "mrr": "0.559", "ndcg": "0.627"}
   {"system": "dense", "k": 20, "recall": "0.933", "precision": "0.047", "hit": "0.933", "mrr": "0.567", "ndcg": "0.653"}
   {"system": "sparse", "k": 1, "recall": "0.617", "precision": "0.617", "hit": "0.617", "mrr": "0.617", "ndcg": "0.617"}
   {"system": "sparse", "k": 3, "recall": "0.800", "precision": "0.267", "hit": "0.800", "mrr": "0.692", "ndcg": "0.719"}
   {"system": "sparse", "k": 5, "recall": "0.917", "precision": "0.183", "hit": "0.917", "mrr": "0.718", "ndcg": "0.767"}
   {"system": 

## 10. Figures (per "Visualization Code" and "Thesis Tables")

Thesis-ready PNGs: per-system retrieval, recall@k curve, overlap/complementarity, and recall by category.


In [11]:
figs = FIG.make_all(agg=agg, per_query=retrieval, items=items)
for name, p in figs.items():
    print(f"{name:<14} {p}")


retrieval      <repo>/notebooks/data/figures/fig1_retrieval.png
recall_curve   <repo>/notebooks/data/figures/fig2_recall_curve.png
ablation       <repo>/notebooks/data/figures/fig3_ablation.png
fine_tuning    <repo>/notebooks/data/figures/fig4_fine_tuning.png
overlap        <repo>/notebooks/data/figures/fig6_overlap.png
category       <repo>/notebooks/data/figures/fig5_category.png


## 11. Reproducibility report (per "Reproducibility")

Git SHA, platform, Python, all models + versions, seeds, metric list.

In [12]:
report = RP.build_report(items=items, cfg=cfg)
p = RP.save(report)
print("report saved ->", p)
print("git:", report.get("git"))
print("models:", json.dumps(report["models"], indent=2))


report saved -> <repo>/notebooks/data/evaluation/reports/reproducibility.json
git: {'commit': 'b684a34d27803470453225a7ceba671d90b7ed6c', 'dirty': True, 'branch': 'feat/rag-ingestion-pipeline'}
models: {
  "generator_llm": "qwen3.8:27b",
  "judge_llm": "qwen3.8:27b",
  "embedding_graphrag": "bge-m3",
  "semantic_similarity_embedding": "BAAI/bge-m3",
  "dense_base": "all-MiniLM-L6-v2",
  "dense_finetuned": [
    "notebooks/data/retrieval/finetuned_stage1/adapter",
    "notebooks/data/retrieval/finetuned_stage2/adapter"
  ],
  "scenario_protocol": {
    "drafters": [
      "gemma3:27b",
      "mistral-small3.1:24b"
    ],
    "adjudicator": "qwen3.8:27b",
    "escalator": "nemotron-3-nano:30b"
  }
}


## 12. Constructed compliance scenarios (per the "Additional information" section)

Compensates for the lack of expert legal review. Two construction strategies:

1. **Synthetic** - formal constraint edits with *correct-by-construction* labels.
2. **LLM-drafted + adjudicated** - Gemma 3 27B -> Candidate A, Mistral Small
   3.1 24B -> Candidate B (anonymized), judged by a blind Qwen 3.8 27B
   adjudicator; a dispute (neither qualifies) escalates to Nemotron-3-nano 30B.
   No drafter ever judges its own draft.


In [13]:
corpus = load_corpus()
SYNTH_PER_DOC = 2     # scale knob: 2 = compact, 3 = full
synth = SC.build_synthetic_scenarios(corpus, per_doc=SYNTH_PER_DOC)
print("synthetic scenarios:", len(synth))
print("labels:", Counter(s.label for s in synth))


synthetic scenarios: 84
labels: Counter({'non_compliant': 42, 'compliant': 42})


### 12b. LLM-drafted + blind-adjudicated set (>LLM, cached; the long step)

In [14]:
VOTED = 12           # scale knob: 12 = compact, 24 = recommended
print("building voted scenarios (Gemma draft A + Mistral draft B -> blind Qwen "
      "adjudicator; Nemotron escalates disputes only) ...")
voted = SC.build_voted_scenarios(corpus, max_scenarios=VOTED)
print("voted scenarios:", len(voted))
print("labels:", Counter(s.label for s in voted))
for s in voted[:3]:
    print(f"  {s.scenario_id} [{s.label}] path={s.metadata.get('decision_path')}")


building voted scenarios (Gemma draft A + Mistral draft B -> blind Qwen adjudicator; Nemotron escalates disputes only) ...


voted scenarios: 12
labels: Counter({'compliant': 6, 'non_compliant': 6})
  vote_001 [compliant] path=single_qualify
  vote_002 [compliant] path=single_qualify
  vote_003 [compliant] path=single_qualify


## 13. Scenario evaluation + four RAG baselines

Per scenario: (a) audit verdict + correction quality, (b) does each RAG
baseline retrieve the responsible provision (hit/rank). Retrieval-only; labels are model-free.


In [15]:
scen_all = synth + voted

# (a) audit + correction + retrieval vs baselines (base encoder)
base_retr = BL.baseline_retrievals(scen_all, engine=eng, k=5, encoder="base")
rows = SC.run_scenario_evaluation(scen_all, retrievals=base_retr)
det = SC.detection_accuracy(rows)
agg_scen = json.loads((E.OUT_GRAPHS / "scenario_agg.json").read_text())
print("scenario rows:", len(rows), "| detection:", det)
print("cites_responsible:", agg_scen.get("cites_responsible"))

# hit-rate per baseline: does top-5 contain the responsible provision?
target_of = {s.scenario_id: s.provision_lineage_id for s in scen_all}
hr = {}
for sid, per_b in base_retr.items():
    if sid not in target_of:
        continue
    t = target_of[sid]
    for bname, lids in per_b.items():
        e = hr.setdefault(bname, {"hit": 0, "n": 0})
        e["n"] += 1
        if t in (lids or []):
            e["hit"] += 1
print("\nhit-rate@5 per baseline:")
for bname, e in sorted(hr.items()):
    print(f"  {bname:<18} {e['hit']/max(e['n'],1):.2f}  ({e['hit']}/{e['n']})")


scenario rows: 96 | detection: {'accuracy': 0.6979166666666666, 'n': 96, 'n_total': 96}
cites_responsible: 1.0



hit-rate@5 per encoder x baseline:
  base             rag_contextual     1.00  (96/96)
  base             rag_dense_fixed    0.70  (67/96)
  base             rag_dense_late     0.72  (69/96)
  base             rag_hybrid_graph   0.94  (90/96)
  finetuned_stage1 rag_contextual     1.00  (96/96)
  finetuned_stage1 rag_dense_fixed    0.71  (68/96)
  finetuned_stage1 rag_dense_late     0.71  (68/96)
  finetuned_stage1 rag_hybrid_graph   0.94  (90/96)
  finetuned_stage2 rag_contextual     1.00  (96/96)
  finetuned_stage2 rag_dense_fixed    0.70  (67/96)
  finetuned_stage2 rag_dense_late     0.71  (68/96)
  finetuned_stage2 rag_hybrid_graph   0.94  (90/96)


## 14. Headline results

A compact summary of the key numbers for the thesis.

In [16]:
print("="*70)
print("RETRIEVAL recall@k (by system) - from aggregate")
for k in (1, 3, 5, 10):
    line = f"  k={k:<3} "
    for sysn in ["sparse","dense","hybrid","graph"]:
        v = next((x for x in agg if x["system"]==sysn and x["metric"]=="recall" and x["k"]==k), None)
        line += f"{sysn}={v['value']:.3f}  " if v else f"{sysn}=NA "
    print(line)
print("\nANSWER (from Table B)")
for r in tB:
    print("  %-10s em=%s f1=%s cites=%s judge_f=%s" %
          (r["system"], r["em"], r["f1"], r["cites_target"], r["judge_faithfulness"]))
print("\nFINE-TUNING A/B (from Table D)")
for r in tD:
    print("  %-16s recall@10 = %s" % (r["variant"], r["recall"]))
print("\nSCENARIOS")
print("  synthetic:", len(synth), "| voted:", len(voted))
print("  detection accuracy:", det)
print("\nLEAKAGE (per Avoid Data Leakage)")
print(" ", leak["counts"])
print("="*70)


RETRIEVAL recall@k (by system) - from aggregate
  k=1   sparse=0.617  dense=0.400  hybrid=0.583  graph=0.650  
  k=3   sparse=0.800  dense=0.700  hybrid=0.833  graph=0.817  
  k=5   sparse=0.917  dense=0.800  hybrid=0.917  graph=0.917  
  k=10  sparse=0.933  dense=0.833  hybrid=0.933  graph=0.933  

ANSWER (from Table B)
  sparse     em=0.000 f1=0.600 cites=1.00 judge_f=4.85
  dense      em=0.000 f1=0.590 cites=1.00 judge_f=4.90
  hybrid     em=0.000 f1=0.625 cites=1.00 judge_f=4.80
  neo4j      em=0.000 f1=0.378 cites=1.00 judge_f=4.73

FINE-TUNING A/B (from Table D)
  dense            recall@10 = 0.833
  dense_ft_s1      recall@10 = 0.883
  dense_ft_s2      recall@10 = 0.883

SCENARIOS
  synthetic: 84 | voted: 12
  detection accuracy: {'accuracy': 0.6979166666666666, 'n': 96, 'n_total': 96}

LEAKAGE (per Avoid Data Leakage)
  {'question_verbatim_in_training': 2, 'targets_that_are_training_positives': 49, 'reference_text_in_training_positives': 49, 'n_queries': 60}
